# 03b — AI Search Attacker Baselines

**RAKSHAK-ICS** | Week 3 — AI Search-Based Attacker Evaluation (Units I–II)

This notebook evaluates 4 AI search attackers against the placeholder Blue Agent:

| # | Attacker | Algorithm | AIML Unit |
|---|----------|----------|-----------|
| 1 | Random | Uniform random perturbations (BFS analogue) | Unit I |
| 2 | IDDFS | Iterative deepening DFS, depth-limited | Unit I |
| 3 | A* Search | Priority queue with disruption heuristic | Unit I |
| 4 | Alpha-Beta | Adversarial game tree, depth=3, pruning | Unit II |

Also demonstrates the **Genetic Algorithm** for reward weight optimization (Unit II).

> **Note**: Uses a placeholder Blue Agent (deviation-based). Real LSTM-AE + GAT fusion comes in Week 5.

In [ ]:
import os
os.chdir('..')

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

plt.style.use('dark_background')
PALETTE = ['#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c084fc', '#ff922b']
sns.set_palette(PALETTE)

print('Setup complete.')

## 1. Load Sensor Data

In [ ]:
from src.preprocess import load_processed_data
from src.ai_search import (
    ATTACKER_REGISTRY, get_attacker, run_all_attackers,
    placeholder_blue_agent, GAOptimizer
)

data = load_processed_data('data/proof/')
X_test = data['X_test']

# Use last timestep of each window as current readings
sensor_data = X_test[:, -1, :] if X_test.ndim == 3 else X_test
print(f'Sensor data: {sensor_data.shape}')
print(f'Available attackers: {list(ATTACKER_REGISTRY.keys())}')

## 2. Run All Attackers (Multi-Seed)

In [ ]:
results = run_all_attackers(
    sensor_data=sensor_data,
    blue_agent_fn=placeholder_blue_agent,
    n_samples=50,
    seeds=[42, 123, 456, 789, 1024],
    max_steps=200,
)

print('\nAll attackers complete!')

## 3. Attacker Comparison Table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Attacker': name.replace('_', ' ').title(),
        'Evasion Rate': r['evasion_rate']['formatted'],
        'Disruption': r['disruption']['formatted'],
        'Reward': r['reward']['formatted'],
    })

df_attackers = pd.DataFrame(rows)
print(df_attackers.to_string(index=False))

## 4. Evasion Rate Comparison

In [ ]:
names = list(results.keys())
evasion_means = [results[n]['evasion_rate']['mean'] for n in names]
evasion_stds = [results[n]['evasion_rate']['std'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Evasion rate
axes[0].bar(range(len(names)), evasion_means, yerr=evasion_stds, capsize=5,
            color=PALETTE[:len(names)], edgecolor='white', linewidth=0.5)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels([n.replace('_', ' ').title() for n in names], fontsize=11)
axes[0].set_ylabel('Evasion Rate', fontsize=13)
axes[0].set_title('Attacker Evasion Rates', fontsize=16, fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(evasion_means, evasion_stds)):
    axes[0].text(i, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

# Disruption
disrupt_means = [results[n]['disruption']['mean'] for n in names]
disrupt_stds = [results[n]['disruption']['std'] for n in names]
axes[1].bar(range(len(names)), disrupt_means, yerr=disrupt_stds, capsize=5,
            color=PALETTE[:len(names)], edgecolor='white', linewidth=0.5)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels([n.replace('_', ' ').title() for n in names], fontsize=11)
axes[1].set_ylabel('Mean Disruption', fontsize=13)
axes[1].set_title('Attacker Disruption Levels', fontsize=16, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(disrupt_means, disrupt_stds)):
    axes[1].text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/ai_attackers_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Single Attack Visualization

In [ ]:
# Run a single A* attack and visualize the perturbation
sample_idx = 100
original = sensor_data[sample_idx]

attacker = get_attacker('a_star', max_steps=200)
result = attacker.attack(original, seed=42)

perturbed = result['perturbed_readings']
delta = perturbed - original

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Original vs perturbed
x = np.arange(len(original))
axes[0].plot(x, original, color=PALETTE[0], alpha=0.8, label='Original', linewidth=1.5)
axes[0].plot(x, perturbed, color=PALETTE[1], alpha=0.8, label='Perturbed (A*)', linewidth=1.5)
axes[0].set_xlabel('Sensor Index', fontsize=12)
axes[0].set_ylabel('Scaled Value [0,1]', fontsize=12)
axes[0].set_title(f'A* Attack: Original vs Perturbed Readings (evasion={result["evasion"]})', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Perturbation magnitude
colors = [PALETTE[1] if d > 0 else PALETTE[3] for d in delta]
axes[1].bar(x, delta, color=colors, alpha=0.8)
axes[1].set_xlabel('Sensor Index', fontsize=12)
axes[1].set_ylabel('Perturbation (\u0394)', fontsize=12)
axes[1].set_title('Per-Sensor Perturbation Magnitude', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0, color='white', linewidth=0.5)

plt.tight_layout()
plt.savefig('results/figures/a_star_attack_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Disruption: {result["disruption"]:.4f}')
print(f'Detected: {result["detected"]}')
print(f'Nodes expanded: {result["nodes_expanded"]}')
print(f'Time: {result["time_elapsed"]:.3f}s')

## 6. GA Reward Weight Optimization (Demo)

In [ ]:
# Demo GA: optimize reward weights for attack effectiveness
def demo_fitness(weights):
    """Fitness = average reward of Random attacker with given weights."""
    lambda1, lambda2, lambda3 = weights
    attacker = get_attacker('random', max_steps=20)
    
    rewards = []
    for i in range(10):
        idx = np.random.randint(len(sensor_data))
        result = attacker.attack(sensor_data[idx], seed=i)
        delta = np.sum(np.abs(result['perturbed_readings'] - sensor_data[idx]))
        detected = 1.0 if result['detected'] else 0.0
        reward = lambda1 * delta - lambda2 * detected - lambda3 * (delta ** 2)
        rewards.append(reward)
    return float(np.mean(rewards))

ga = GAOptimizer(
    fitness_fn=demo_fitness,
    population_size=10,
    n_generations=15,
    tournament_size=3,
    mutation_std=0.1,
    seed=42,
)

best = ga.evolve()
print(f'\nBest reward weights: \u03bb1={best.genes[0]:.3f}, \u03bb2={best.genes[1]:.3f}, \u03bb3={best.genes[2]:.3f}')
print(f'Best fitness: {best.fitness:.4f}')

## 7. GA Convergence Curve

In [ ]:
gens = [h['generation'] for h in ga.history]
best_fit = [h['best_fitness'] for h in ga.history]
mean_fit = [h['mean_fitness'] for h in ga.history]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(gens, best_fit, 'o-', color=PALETTE[0], linewidth=2, markersize=5, label='Best Fitness')
ax.plot(gens, mean_fit, 's--', color=PALETTE[1], linewidth=1.5, markersize=4, label='Mean Fitness', alpha=0.7)
ax.fill_between(gens, mean_fit, best_fit, alpha=0.15, color=PALETTE[0])
ax.set_xlabel('Generation', fontsize=13)
ax.set_ylabel('Fitness', fontsize=13)
ax.set_title('GA Convergence: Reward Weight Optimization', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/ga_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Attacker Results

In [ ]:
import json

os.makedirs('results/tables', exist_ok=True)
with open('results/tables/ai_attacker_results.json', 'w') as f:
    json.dump(results, f, indent=2)

df_attackers.to_csv('results/tables/ai_attacker_results.csv', index=False)
print('Saved: results/tables/ai_attacker_results.json')
print('Saved: results/tables/ai_attacker_results.csv')

## 9. Key Findings

### Summary
- **4 AI search attackers** evaluated against placeholder Blue Agent
- **A*** and **Alpha-Beta** are expected to outperform Random and IDDFS
- **GA optimizer** successfully converges on reward weights {\u03bb1, \u03bb2, \u03bb3}
- These results will be re-evaluated against the real LSTM-AE + GAT Blue Agent (Week 5)

### AIML Curriculum Coverage
- **Unit I**: BFS (Random), DFS (IDDFS), A* (informed search)
- **Unit II**: Alpha-Beta (adversarial game), GA (evolutionary optimization)